# What is practice worth?

Two questions, both answerable now that the model solves in seconds.

1. If I get more accurate, how much better do I actually get?
2. Should I practise doubles or trebles?

## Choosing the model

A club match is a race in **visits**, so that is the currency: the 3-dart MDP
with the minimum-visits objective. It solves in about ten seconds per ability,
which makes a dense sweep affordable, and notebook 04 established it is within
about 1.4 points of win probability of the full two-player game.

The second question needs something the model did not previously support:
accuracy that depends on **where you are aiming**. That cannot go through a
single FFT, because every aiming point would need its own kernel. It does not
have to -- partition the aiming grid into doubles, trebles and everything else,
and run one FFT per class.

In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('..', '..'))
if module_path not in sys.path:
    sys.path.append(module_path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25,
    'grid.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 11, 'font.size': 9, 'legend.frameon': False,
})

from darts import players
from darts.mdp_3turn import ThreeDartMDP
from darts.practice import (classify_points, practice_split_value,
                            sigma_sensitivity, transition_arrays_by_class)
from darts.transitions import transition_arrays

SIGMAS = np.arange(6.0, 30.1, 1.0)
visits = []
for sigma in SIGMAS:
    tr = transition_arrays(players.BOARD_PIXELS, float(sigma), point_stride=4)
    m = ThreeDartMDP(tr['probs'], tr['checkout_probs'], tr['allowed_scores'],
                     players.GAME_START, dart_cost=0.0, turn_cost=1.0).solve()
    visits.append(-m.V1[players.GAME_START])
visits = np.array(visits)
print(f'solved {len(SIGMAS)} abilities')

In [ ]:
d_visits = sigma_sensitivity(SIGMAS, visits)
avg = 3 * players.GAME_START / (visits * 3)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
axes[0].plot(SIGMAS, visits, color='#2b6cb0', lw=1.8)
axes[0].set_xlabel(r'throwing accuracy $\sigma$ (mm)')
axes[0].set_ylabel('visits to finish 501')
axes[0].set_title('Where you are')

axes[1].plot(SIGMAS, d_visits, color='#2b6cb0', lw=1.8)
axes[1].set_xlabel(r'throwing accuracy $\sigma$ (mm)')
axes[1].set_ylabel('visits saved per mm of improvement')
axes[1].set_title('What one millimetre is worth')
fig.tight_layout()

rows = [{'band': b, 'sigma': s,
         'visits': round(float(np.interp(s, SIGMAS, visits)), 2),
         '3-dart average': round(3 * 501 / (3 * np.interp(s, SIGMAS, visits)), 1),
         'visits saved per mm': round(float(sigma_sensitivity(SIGMAS, visits, at=s)), 3)}
        for b, s in players.ABILITY_BANDS.items()]
pd.DataFrame(rows).set_index('band')

The return on accuracy is strongly non-linear. A pub player gains about half a
visit per leg for each millimetre they tighten up; an elite player gains a tenth
of that. Improvement is cheapest when you are worst -- which is the opposite of
how it feels, because a beginner's millimetre is hard-won.

## Doubles or trebles?

The comparison has to be like for like: the same *proportional* gain in
accuracy, applied to different targets.

In [ ]:
rows = []
for band in ['county', 'league', 'club', 'pub']:
    for r in practice_split_value(players.ABILITY_BANDS[band], improvement=0.2,
                                  point_stride=4):
        rows.append({'band': band, **r})
split = pd.DataFrame(rows)
split.pivot(index='band', columns='scenario', values='visits saved')[
    ['doubles only', 'trebles only', 'everything']].reindex(
    ['county', 'league', 'club', 'pub'])

In [ ]:
piv = split.pivot(index='band', columns='scenario',
                  values='visits saved').reindex(['county', 'league', 'club', 'pub'])
fig, ax = plt.subplots(figsize=(7.5, 3.4))
w = 0.35
x = np.arange(len(piv))
ax.bar(x - w/2, piv['doubles only'], w, label='practise doubles', color='#93c5fd')
ax.bar(x + w/2, piv['trebles only'], w, label='practise trebles', color='#2b6cb0')
ax.set_xticks(x); ax.set_xticklabels(piv.index)
ax.set_ylabel('visits saved per leg')
ax.set_title('A 20% tighter group, spent on one thing or the other')
ax.legend()
fig.tight_layout()

**Trebles win, at every ability.** That is the opposite of the usual advice.

The reason is arithmetic rather than psychology: a leg is roughly seven visits
of scoring and one of doubling. Improving the seven is worth more than
improving the one, even though the one is the part that feels decisive and is
the part you remember losing.

Two caveats worth stating rather than burying. This assumes accuracy transfers
within a class of target, and it assumes your doubles accuracy is the same as
your trebles accuracy to begin with -- which for a player with a genuine
doubles problem is exactly the assumption that fails. The model can represent
that case; it just needs the per-class sigmas measured rather than assumed,
which is what notebook 07 is for.

## What a millimetre is worth in a match

In [ ]:
from darts.match import FORMATS, match_win_probability
from darts.practice import leg_win_probability

base = players.ABILITY_BANDS['league']
opp_visits = float(np.interp(base, SIGMAS, visits))
rows = []
for delta in [0.0, 0.5, 1.0, 2.0, 3.0]:
    my_visits = float(np.interp(base - delta, SIGMAS, visits))
    p1 = leg_win_probability(my_visits, opp_visits, throws_first=True)
    p2 = leg_win_probability(my_visits, opp_visits, throws_first=False)
    rows.append({'sigma improvement (mm)': delta,
                 'my visits': round(my_visits, 2),
                 'P(win leg, throwing)': round(p1, 4),
                 'P(win best of 11)': round(
                     match_win_probability(p1, p2, **FORMATS['best of 11 legs (Premier League, UK Open early)']), 4)})
pd.DataFrame(rows).set_index('sigma improvement (mm)')

## Summary

* One millimetre of accuracy is worth roughly 0.35 visits per leg to a league
  player and half a visit to a pub player; the return shrinks as you improve.
* **Practising trebles beats practising doubles at every ability level**, because
  a leg is mostly scoring.
* The exception is a player whose doubles are specifically worse than their
  scoring -- and that is measurable rather than assumable.